# 📦 1️⃣ Importaciones y carga de claves

In [1]:
import os
import json
import numpy as np
from pathlib import Path
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

# -----------------------------
# RUTAS CORRECTAS PARA JUPYTER
# -----------------------------

# Ruta del proyecto (sube 1 nivel desde /notebooks)
BASE_DIR = Path().resolve().parents[0]      # <-- esta es la clave

# Si tu notebook está en: 03_IA_Segunda_Guerra_Mundial/notebooks/
# Entonces BASE_DIR apunta a: 03_IA_Segunda_Guerra_Mundial/

PROCESSED_DIR = BASE_DIR / "data" / "processed"
CHUNKS_PATH = PROCESSED_DIR / "ww2_chunks_plus_qa.jsonl"
EMB_PATH = PROCESSED_DIR / "ww2_embeddings_plus_qa.npy"


# 📄 2️⃣ Cargar chunks y embeddings

In [2]:
def cargar_datos():
    chunks = []
    with open(CHUNKS_PATH, "r", encoding="utf-8") as f:
        for line in f:
            chunks.append(json.loads(line))

    emb_matrix = np.load(EMB_PATH)

    print(f"Chunks cargados: {len(chunks)}")
    print(f"Embeddings shape: {emb_matrix.shape}")

    return chunks, emb_matrix


chunks, emb_matrix = cargar_datos()


Chunks cargados: 1510
Embeddings shape: (1510, 3072)


# 🧠 3️⃣ Función para crear embeddings de la pregunta

In [3]:
def embed_text(text):
    resp = client.embeddings.create(
        model="text-embedding-3-large",
        input=text,
    )
    return np.array(resp.data[0].embedding, dtype="float32")


# 📏 4️⃣ Similitud coseno + recuperación de top-k fragmentos

In [4]:
def cosine_sim(query_vec, docs_matrix):
    num = docs_matrix @ query_vec
    den = (np.linalg.norm(docs_matrix, axis=1) * np.linalg.norm(query_vec) + 1e-10)
    return num / den


def recuperar_chunks(query, k=5):
    q_emb = embed_text(query)
    sims = cosine_sim(q_emb, emb_matrix)

    idx = np.argsort(sims)[::-1][:k]

    resultados = []
    for i in idx:
        c = chunks[int(i)]
        resultados.append({
            "score": float(sims[i]),
            "text": c["text"],
            "source": c["source"],
            "page": c["page"]
        })

    return resultados


# 📝 5️⃣ Prompt conciso + llamada al modelo

In [5]:
def generar_respuesta(query, contexto):
    # Construir fragmentos
    partes = []
    for i, c in enumerate(contexto):
        partes.append(
            f"[Fragmento {i+1} | Fuente: {c['source']} pág. {c['page']}]\n{c['text']}"
        )
    context_text = "\n\n".join(partes)

    mensajes = [
        {
            "role": "system",
            "content": (
                "Eres un historiador experto en la Segunda Guerra Mundial. "
                "Respondes siempre en español, de manera breve y directa, máximo 6 líneas. "
                "Tu única fuente es el contexto proporcionado. No inventes datos."
            ),
        },
        {
            "role": "user",
            "content": (
                f"{context_text}\n\n"
                "Usa SOLO la información anterior. Si algo no aparece, responde: "
                "\"No existe información suficiente en los documentos para responder.\"\n\n"
                "Instrucciones:\n"
                "- Máximo 6 líneas.\n"
                "- Sin subtítulos.\n"
                "- Respuesta clara y concisa.\n"
                "- Añade citas al final: (Fuente: <PDF>, pág. X).\n\n"
                f"Pregunta: {query}"
            ),
        },
    ]

    resp = client.chat.completions.create(
        model="gpt-4.1-mini",
        messages=mensajes,
        max_completion_tokens=300,
    )

    return resp.choices[0].message.content


# ❓ 6️⃣ Función final para preguntarle:

In [6]:
def responder(query, k=5):
    contexto = recuperar_chunks(query, k)
    respuesta = generar_respuesta(query, contexto)
    return respuesta


# PREGUNTAR

In [7]:
responder("¿Por qué fue importante la Operación Barbarroja?")


'La Operación Barbarroja fue clave porque representó la invasión alemana a la Unión Soviética con el objetivo ideológico y económico de conquistar territorios para repoblarlos y obtener recursos estratégicos como el petróleo del Cáucaso y la agricultura de Ucrania. Pretendía destruir al Estado soviético para obtener espacio vital (lebensraum) y esclavizar o eliminar a sus pueblos. Sin embargo, a pesar de las victorias iniciales, se estancó y marcó el fracaso del plan rápido del Eje, iniciando una guerra de desgaste que Alemania no pudo sostener (Fuente: Operación_Barbarroja.pdf, págs. 0, 1, 28).'

In [8]:
responder("¿Cuáles fueron las principales causas de la Segunda Guerra Mundial?")

'Las principales causas de la Segunda Guerra Mundial fueron la inestabilidad generada por el Tratado de Versalles, la crisis económica de entreguerras, la política expansionista y racista de Hitler, y la incapacidad de las potencias para contener la agresión alemana. Además, las tensiones no resueltas tras la Primera Guerra Mundial y las rivalidades entre las naciones europeas contribuyeron al conflicto. También influyeron la combinación de nacionalismos extremos y el miedo al comunismo (Fuente: qa_manual, pág. 0; Segunda_Guerra_Mundial.pdf, pág. 1; qa_manual, pág. 0).'

In [9]:
responder("¿Cuándo empezó la segunda guerra mundial?")

'La Segunda Guerra Mundial comenzó oficialmente el 1 de septiembre de 1939 con la invasión alemana de Polonia. A las 4:26 horas de esa madrugada, bombardeos alemanes y ataques comenzaron, como el incidente de Gliwice y el fuego del acorazado Schleswig-Holstein. Dos días después, Reino Unido y Francia declararon la guerra a Alemania (Fuente: Segunda_Guerra_Mundial.pdf, págs. 0,1,15).'

In [10]:
responder("¿Cómo afectó la relación entre Hitler y sus generales al desarrollo de la guerra?")

'La relación entre Hitler y sus generales se deterioró por la desconfianza mutua y la intervención autoritaria de Hitler. Este asumió el mando directo, desoyendo a asesores experimentados como Gerd von Rundstedt y Erich von Manstein, lo que redujo la eficacia militar y la capacidad de adaptación en el frente. Los cambios constantes en los objetivos y la falta de autonomía operativa dificultaron la coordinación bélica alemana. (Fuente: Operación_Barbarroja.pdf, pág. 59; qa_manual, pág. 0).'